# Building a Debugging Assistant

Learning objective: Build a local Gemma 4 assistant that teaches structured debugging, verification, and reflection.

The model does not give final corrected code. It helps you build a debugging habit:

1. **Observe** — What error or output do I see?
2. **Root Cause** — What might be causing the bug?
3. **Check** — What should I inspect next?
4. **Verify** — Did my fix actually work?
5. **Reflect** — What will I check next time?

![Guided debugging workflow](debugging_workflow.png)

The main idea is **verification before trust**. The model can suggest a root cause, but Python runs the check and you verify the result.


## Why this still matters

AI can write code quickly. That is useful.

But speed is not the same as correctness.

| AI can help with... | You still need to check... |
|---|---|
| Writing a first draft | Did it solve the right problem? |
| Explaining a possible bug | Is the explanation supported by evidence? |
| Suggesting a fix | Did we test the fix? |
| Reading an image or report | Did it extract the details correctly? |
| Calling a tool | Did the program run the tool safely? |

In this notebook, the base output may sound helpful. The goal is to learn how to check it.

**Goal:** learn how to say, "This answer looks helpful, but I know how to verify it."


## Step 1: Imports

Run the setup imports first. These libraries load the model, read JSON, run small checks, call the optional Gemini API, and display simple widgets.


In [20]:
# JUST RUN THIS CELL
from llama_cpp import Llama  # loads local GGUF models
import os  # joins folder and file names
import json  # reads JSON text as Python data
import subprocess  # runs one small diagnostic program
import sys  # finds the current Python program
import time  # measures how long model calls take
import pandas as pd  # makes result tables
from IPython.display import display  # displays widgets
import ipywidgets as widgets  # creates dropdown menus

try:
    from dotenv import load_dotenv  # reads API keys from a .env file
except Exception as error:
    print("python-dotenv is not installed:", error)

try:
    from google import genai  # calls hosted Gemma models through the Gemini API
    from google.genai import types  # builds Gemini API config objects
except Exception as error:
    print("google-genai is not installed:", error)


## Step 2: Load the local model

This notebook uses **Gemma 4 E2B Instruct** in GGUF format.

GGUF is a local model file format used by llama.cpp tools.

Gemma 4 E2B is small enough for a local classroom demo. The optional extensions use hosted **Gemma 4 31B** through the Gemini API.


In [21]:
# RUN THIS CELL
model_directory = "/Users/seohachoi/models"
model_name = "gemma-4-e2b-it-edited-q4_0.gguf"

# Create the full path to the model.
model_path = os.path.join(model_directory, model_name)

# Load the model using llama-cpp-python.
# n_ctx: context window size, or how much text the model can see at once.
# n_threads: number of CPU threads. None means auto-detect.
# verbose: whether to print loading information.
model = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=None,
    verbose=False,
)

print()
print(f"Model loaded successfully: {model_name}")


llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512



Model loaded successfully: gemma-4-e2b-it-edited-q4_0.gguf


## Step 3: Get base output

First, ask the model for debugging help with only a light prompt. This shows what can happen before we add stricter instructions.


In [22]:
# RUN THIS CELL
base_prompt = """
My Python code does not work. Please help me debug it, but do not give me the final corrected code.

Problem:
I want to write a function that sums the squares from 0 through n.

Expected:
The sum should include n in the squared terms.

Actual:
The current result does not include n.

Code:
def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total

Tried:
I checked the formula, but I did not check the exact values generated by the loop.
""".strip()

print(base_prompt)


My Python code does not work. Please help me debug it, but do not give me the final corrected code.

Problem:
I want to write a function that sums the squares from 0 through n.

Expected:
The sum should include n in the squared terms.

Actual:
The current result does not include n.

Code:
def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total

Tried:
I checked the formula, but I did not check the exact values generated by the loop.


Send the base prompt to the model.

Look for this question: does the model guide the student, or does it jump straight to final code?


In [23]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
messages = [
    {"role": "user", "content": base_prompt}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=512,
    temperature=0.1,
)

base_output = response["choices"][0]["message"]["content"]
print(base_output)


This is a great start! You've correctly identified the goal: summing the squares of numbers from 0 up to and including $n$.

Let's look closely at how you are using the `np.arange` function in your loop.

1.  **Understanding `np.arange(start, stop, step)`:** This function generates a sequence of numbers starting at `start`, incrementing by `step`, and stopping *before* it reaches `stop`.

2.  **Analyzing your loop:**
    ```python
    for i in np.arange(0, n, 1):
    ```
    If $n=5$, `np.arange(0, 5, 1)` will generate the sequence: $0, 1, 2, 3, 4$. It stops *before* 5.

3.  **The Requirement:** You stated that the sum should include $n$ in the squared terms.

**Debugging Question:** How can you adjust the arguments passed to `np.arange` so that the sequence generated includes the number $n$ in the iteration? Think about what the `stop` argument in `np.arange` controls.


### Checkpoint 1

Did the base output follow the request?

Check two things:

1. Did it guide the student to inspect evidence first?
2. Did it give a solution or corrected code even though the prompt asked it not to?

*Double click to type your answer here*


## Step 4: Write the model instructions

Now JSON becomes useful.

| Output style | Good for | Problem |
|---|---|---|
| Plain text | Reading an explanation | Hard for code to reuse |
| JSON | Pulling out specific fields | More rigid to write |

We use JSON here because the notebook needs the same fields every time:

| Field | What the student looks for |
|---|---|
| `likely_cause` | What might be wrong? |
| `next_check` | What should we inspect next? |
| `hint` | How can we think about it without final code? |
| `verify` | How should we test the fix? |

JSON does not make the model automatically correct. It makes the answer easier to inspect, compare, and reuse in code.

We use a low temperature because this lesson needs stable JSON fields.


In [24]:
# RUN THIS CELL
instructions = """
You are a debugging guide for beginning Python students.
Do not give final corrected code.
Use only the student report.

Return valid JSON with these exact fields:
status, known, need, likely_cause, next_check, tool_name, hint, verify, reflect.

If Problem, Expected, Actual, Code, and Tried are present, use status = "ready" and need = "none".
Use status = "need" only if one of those sections is missing.

For ready cases in this notebook, use tool_name = "run_check".
Use tool_name = "" only when status = "need".
Do not write code for the tool.
The notebook has only one allowed tool name: run_check.
The notebook will choose the prepared classroom check.

Write next_check as something the student should inspect or test next.
Do not give the exact final code edit.
Do not name the exact replacement expression.

If a tool result is provided, use it as evidence.
Do not repeat the previous JSON unchanged.

Do not reveal hidden chain-of-thought.
""".strip()

json_mode = {"type": "json_object"}

print(instructions)

You are a debugging guide for beginning Python students.
Do not give final corrected code.
Use only the student report.

Return valid JSON with these exact fields:
status, known, need, likely_cause, next_check, tool_name, hint, verify, reflect.

If Problem, Expected, Actual, Code, and Tried are present, use status = "ready" and need = "none".
Use status = "need" only if one of those sections is missing.

For ready cases in this notebook, use tool_name = "run_check".
Use tool_name = "" only when status = "need".
Do not write code for the tool.
The notebook has only one allowed tool name: run_check.
The notebook will choose the prepared classroom check.

Write next_check as something the student should inspect or test next.
Do not give the exact final code edit.
Do not name the exact replacement expression.

If a tool result is provided, use it as evidence.
Do not repeat the previous JSON unchanged.

Do not reveal hidden chain-of-thought.


The instructions go in the `system` message. The debugging report goes in the `user` message.

The next cell stores the same output fields as a JSON schema. The hosted Gemma API extension uses this schema later.

The model chooses a tool name only. The notebook decides what code can run.

In [25]:
# JUST RUN THIS CELL
api_json_schema = {
    "type": "object",
    "properties": {
        "status": {"type": "string"},
        "known": {"type": "string"},
        "need": {"type": "string"},
        "likely_cause": {"type": "string"},
        "next_check": {"type": "string"},
        "tool_name": {"type": "string"},
        "hint": {"type": "string"},
        "verify": {"type": "string"},
        "reflect": {"type": "string"}
    },
    "required": ["status", "known", "need", "likely_cause", "next_check", "tool_name", "hint", "verify", "reflect"]
}

print(json.dumps(api_json_schema, indent=2))


{
  "type": "object",
  "properties": {
    "status": {
      "type": "string"
    },
    "known": {
      "type": "string"
    },
    "need": {
      "type": "string"
    },
    "likely_cause": {
      "type": "string"
    },
    "next_check": {
      "type": "string"
    },
    "tool_name": {
      "type": "string"
    },
    "hint": {
      "type": "string"
    },
    "verify": {
      "type": "string"
    },
    "reflect": {
      "type": "string"
    }
  },
  "required": [
    "status",
    "known",
    "need",
    "likely_cause",
    "next_check",
    "tool_name",
    "hint",
    "verify",
    "reflect"
  ]
}


## Step 5: Choose structured input

Now we give the model a structured bug report.

Each case uses the same five parts:

| Section | Meaning |
|---|---|
| Problem | What the student wanted to do |
| Expected | What should have happened |
| Actual | What happened instead |
| Code | The code being debugged |
| Tried | What the student already checked |

This makes the model less likely to guess and makes the answers easier to compare.

Each case also has one prepared classroom check for the tool step.


In [26]:
# JUST RUN THIS CELL
test_cases = [
    {
        "name": "Loop boundary",
        "problem": "I want to write a function that sums squares from 0 through n.",
        "expected": "I expect the sum to include n in the squared terms.",
        "actual": "I see that the current result does not include n.",
        "code": """def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total""",
        "tried": "I checked the formula, but I did not check the exact values generated by the loop.",
        "check_code": """print("Values generated by np.arange(0, 3, 1):")
print(np.arange(0, 3, 1))""",
    },
    {
        "name": "Table column",
        "problem": "I want to create a num_tags column with one value per row.",
        "expected": "I expect each row to count the tags in that row.",
        "actual": "I see that the code counts the word tags instead of the column values.",
        "code": """def count_tags(string):
    return len(string.split(" "))

videos = Table().with_columns(
    "creator", make_array("a", "b", "c"),
    "tags", make_array("#data #python", "#study", "#fun #data")
)

videos = videos.with_column("num_tags", count_tags("tags"))""",
        "tried": "I tried using with_column because I want to add a new column.",
        "check_code": """def count_tags(string):
    return len(string.split(" "))

print("count_tags(\"tags\"):", count_tags("tags"))
print("count_tags(\"#data #python\"):", count_tags("#data #python"))""",
    },
    {
        "name": "Simulation deck",
        "problem": "I want to simulate a game with cards 1, 2, 3, and 4.",
        "expected": "I expect each game to draw from a four-card deck.",
        "actual": "I see that the deck expression leaves out the final card.",
        "code": """def simulate_game(n):
    wins = 0
    for i in np.arange(n):
        deck = np.arange(1, 4)
        cards = np.random.choice(deck, 2, replace=False)
        if cards.item(0) > cards.item(1):
            wins = wins + 1
    return wins / n""",
        "tried": "I inspected the random choice line, but I did not inspect the deck values.",
        "check_code": """print("Deck values from np.arange(1, 4):")
print(np.arange(1, 4))""",
    },
]

case_names = []
for case in test_cases:
    case_names.append(case["name"])

case_dropdown = widgets.Dropdown(
    options=case_names,
    value=case_names[0],
    description="Problem:",
)

display(case_dropdown)


Dropdown(description='Problem:', options=('Loop boundary', 'Table column', 'Simulation deck'), value='Loop bou…

Build the student bug report.

This turns the selected case into the structured input that the model will read.


In [27]:
# RUN THIS CELL AFTER CHOOSING A PROBLEM
selected_case = test_cases[0]
for case in test_cases:
    if case["name"] == case_dropdown.value:
        selected_case = case

prompt = f"""
Please help me debug my code.
Do not give me the final corrected code.

All five report fields are present, so use status = "ready".

Problem:
{selected_case["problem"]}

Expected:
{selected_case["expected"]}

Actual:
{selected_case["actual"]}

Code:
```python
{selected_case["code"]}
```

Tried:
{selected_case["tried"]}
""".strip()

print("Selected case:", selected_case["name"])
print(prompt[:1000])


Selected case: Loop boundary
Please help me debug my code.
Do not give me the final corrected code.

All five report fields are present, so use status = "ready".

Problem:
I want to write a function that sums squares from 0 through n.

Expected:
I expect the sum to include n in the squared terms.

Actual:
I see that the current result does not include n.

Code:
```python
def sum_squares(n):
    total = 0
    for i in np.arange(0, n, 1):
        total = total + i ** 2
    return total
```

Tried:
I checked the formula, but I did not check the exact values generated by the loop.


Ask Gemma 4 E2B for a guided JSON response.

Look for `likely_cause`, `next_check`, `hint`, and `verify`.


In [28]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": prompt}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=600,
    temperature=0.1,
    response_format=json_mode,
)

raw = response["choices"][0]["message"]["content"]
result = json.loads(raw)
print(json.dumps(result, indent=2))


{
  "status": "ready",
  "known": "The loop in your function does not include the value 'n' in the summation.",
  "need": "none",
  "likely_cause": "The `np.arange(0, n, 1)` function generates numbers up to, but not including, `n`.",
  "next_check": "Check the stop value in your `np.arange` call to ensure it includes `n` in the sequence.",
  "tool_name": "run_check",
  "hint": "Review the documentation for `np.arange` regarding its stop condition.",
  "verify": "Test the function with a small value of `n` (e.g., n=3) and manually trace what values `i` takes inside the loop.",
  "reflect": "The issue lies in how the range of numbers is defined in the loop."
}


The model writes JSON text. Python reads that JSON into a dictionary.

The next cell keeps the classroom workflow stable if the model forgets to request the prepared check.


In [29]:
# RUN THIS CELL
all_fields_present = True

for field_name in ["problem", "expected", "actual", "code", "tried"]:
    if selected_case[field_name].strip() == "":
        all_fields_present = False

if all_fields_present:
    result["status"] = "ready"
    result["need"] = "none"

if result["status"] == "ready" and result.get("tool_name", "") == "":
    result["tool_name"] = "run_check"

print(json.dumps(result, indent=2))


{
  "status": "ready",
  "known": "The loop in your function does not include the value 'n' in the summation.",
  "need": "none",
  "likely_cause": "The `np.arange(0, n, 1)` function generates numbers up to, but not including, `n`.",
  "next_check": "Check the stop value in your `np.arange` call to ensure it includes `n` in the sequence.",
  "tool_name": "run_check",
  "hint": "Review the documentation for `np.arange` regarding its stop condition.",
  "verify": "Test the function with a small value of `n` (e.g., n=3) and manually trace what values `i` takes inside the loop.",
  "reflect": "The issue lies in how the range of numbers is defined in the loop."
}


Show the workflow status.

`need` means the model needs more information.

`ready` means the model has enough information to continue.


In [30]:
# RUN THIS CELL
if result["status"] == "need":
    print("Status: need")
    print("Missing information:", result.get("need", ""))
else:
    print("Status: ready")
    print("Known facts:", result.get("known", ""))
    print("Likely cause:", result.get("likely_cause", ""))
    print("Next check:", result.get("next_check", ""))
    print("Hint:", result.get("hint", ""))
    print("Verify:", result.get("verify", ""))
    print("Reflect:", result.get("reflect", ""))


Status: ready
Known facts: The loop in your function does not include the value 'n' in the summation.
Likely cause: The `np.arange(0, n, 1)` function generates numbers up to, but not including, `n`.
Next check: Check the stop value in your `np.arange` call to ensure it includes `n` in the sequence.
Hint: Review the documentation for `np.arange` regarding its stop condition.
Verify: Test the function with a small value of `n` (e.g., n=3) and manually trace what values `i` takes inside the loop.
Reflect: The issue lies in how the range of numbers is defined in the loop.


### Checkpoint 2

Compare the base output from Step 3 with the JSON output from Step 5.

Was JSON useful here?

Check four fields:

1. `likely_cause`: Does it name a possible cause?
2. `next_check`: Does it suggest something to inspect?
3. `hint`: Does it guide without giving final code?
4. `verify`: Does it ask the student to test the fix?

Then decide: did JSON make the answer easier to check, or just more complicated?

*Double click to type your answer here*


## Step 6: Run a diagnostic check

Now tool use becomes useful.

The model can suggest a check, but it does not run Python. The notebook runs Python.

| Step | Who does it? | What happens? |
|---|---|---|
| 1 | Model | Suggests `run_check` |
| 2 | Notebook | Decides whether the tool is allowed |
| 3 | Python | Runs the diagnostic check |
| 4 | Student | Decides whether the evidence supports the fix |

The model suggests. Python produces evidence. The student verifies.

This is a beginner tool loop. The hosted API tool-calling version appears in Extension B.


In [31]:
# RUN THIS CELL
allowed_tool_name = "run_check"
requested_tool_name = result.get("tool_name", "")

# The model only chose a tool name.
# The notebook still decides what code can run.
print("Requested tool:", requested_tool_name)

if requested_tool_name != allowed_tool_name:
    check_result = None
    print("No allowed tool was requested.")
else:
    print("Allowed tool request found.")
    print("The notebook will run this prepared classroom check:")
    print(selected_case["check_code"])

    check_program = """
import numpy as np
""" + "\n" + selected_case["check_code"]

    completed = subprocess.run(
        [sys.executable, "-I", "-c", check_program],
        capture_output=True,
        text=True,
        timeout=3,
    )

    check_result = {
        "ok": completed.returncode == 0,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
    }

    print(json.dumps(check_result, indent=2))


Requested tool: run_check
Allowed tool request found.
The notebook will run this prepared classroom check:
print("Values generated by np.arange(0, 3, 1):")
print(np.arange(0, 3, 1))
{
  "ok": true,
  "stdout": "Values generated by np.arange(0, 3, 1):\n[0 1 2]\n",
  "stderr": ""
}


Send the check result back.

The check result is evidence. The model uses it to update the root cause and next check.

The student still verifies the final fix.

In [32]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
if check_result is None:
    updated = None
    print("No tool result to send back.")
else:
    followup_prompt = f"""
I ran the requested check.

Previous JSON:
{json.dumps(result, indent=2)}

Check result:
{json.dumps(check_result, indent=2)}

Use the check result as evidence.
Update known, likely_cause, next_check, hint, verify, and reflect.
Do not repeat the previous JSON unchanged.
Do not give the exact final code edit.
Return the same JSON fields.
""".strip()

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": followup_prompt}
    ]

    response = model.create_chat_completion(
        messages=messages,
        max_tokens=600,
        temperature=0.1,
        response_format=json_mode,
    )

    updated_raw = response["choices"][0]["message"]["content"]
    updated = json.loads(updated_raw)
    print(json.dumps(updated, indent=2))

{
  "status": "ready",
  "known": "The loop in your function does not include the value 'n' in the summation.",
  "need": "none",
  "likely_cause": "The `np.arange(0, n, 1)` function generates numbers up to, but not including, `n`. For example, if `n=3`, it generates 0, 1, and 2.",
  "next_check": "Adjust the stop value in your `np.arange` call to ensure the sequence includes `n` if that is the desired behavior for the summation.",
  "tool_name": "run_check",
  "hint": "Review the documentation for `np.arange` regarding its stop condition. If you want to include `n`, you might need to use `n + 1` as the stop value.",
  "verify": "Test the function with a small value of `n` (e.g., n=3) and manually trace what values `i` takes inside the loop, comparing it to the output from `np.arange(0, 3, 1)` which was [0, 1, 2].",
  "reflect": "The issue lies in how the range of numbers is defined in the loop, specifically how `np.arange` excludes the stop value."
}


### Checkpoint 3

Was the tool step useful here?

Answer three questions:

1. What evidence did Python show?
2. Could the model know that evidence without running code?
3. Who still decides whether the final fix is correct?

*Double click to type your answer here*


## Step 7: Compare example cases

Run the same prompt pattern on every case and compare the next checks.


In [33]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
rows = []

for case in test_cases:
    case_prompt = f"""
Please help me debug my code.
Do not give me the final corrected code.
All five report fields are present, so use status = "ready".

Problem:
{case["problem"]}

Expected:
{case["expected"]}

Actual:
{case["actual"]}

Code:
```python
{case["code"]}
```

Tried:
{case["tried"]}
""".strip()

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": case_prompt}
    ]

    response = model.create_chat_completion(
        messages=messages,
        max_tokens=600,
        temperature=0.1,
        response_format=json_mode,
    )

    case_raw = response["choices"][0]["message"]["content"]
    case_result = json.loads(case_raw)

    case_result["status"] = "ready"
    case_result["need"] = "none"

    if case_result.get("tool_name", "") == "":
        case_result["tool_name"] = "run_check"

    rows.append({
        "Case": case["name"],
        "Status": case_result.get("status", ""),
        "Likely Cause": case_result.get("likely_cause", ""),
        "Next Check": case_result.get("next_check", ""),
        "Tool": case_result.get("tool_name", ""),
    })

results_table = pd.DataFrame(rows)
results_table

,Case,Status,Likely Cause,Next Check,Tool
0,Loop boundary,ready,"The `np.arange(0, n, 1)` function generates nu...",Check the stop value in your `np.arange` call ...,run_check
1,Table column,ready,The function passed to `with_column` is operat...,Inspect the type of the 'tags' column after it...,run_check
2,Simulation deck,ready,"The `np.arange(1, 4)` function generates numbe...","Check the output of `np.arange(1, 4)` to see w...",run_check


### Checkpoint 4

Which next check is clearest? What makes it useful for a beginner?

*Double click to type your answer here*


## Optional Extension Setup: Hosted Gemma with Gemini API

The core notebook uses local Gemma 4 E2B through llama.cpp.

The optional extensions use the Google GenAI SDK to call hosted Gemma 4 31B through the Gemini API. This is not a separate Gemma-only API.

The hosted model name is **gemma-4-31b-it**.

Before running the setup cell, check that your `.env` file has this line:

`GEMINI_API_KEY=...`

In [34]:
# RUN THIS CELL
if "load_dotenv" in globals():
    load_dotenv()
else:
    print("python-dotenv is not installed. Paste your API key below if needed.")

gemma_api_model = "gemma-4-31b-it"
gemini_api_key = os.getenv("GEMINI_API_KEY")

# Uncomment the next line and paste your key if load_dotenv did not find it.
# gemini_api_key = "PASTE_YOUR_KEY_HERE"

if "genai" not in globals():
    gemini_client = None
    print("google-genai is not installed, so the optional API section cannot run yet.")
elif gemini_api_key:
    gemini_client = genai.Client(api_key=gemini_api_key)
    print("Gemini API client is ready for", gemma_api_model)
else:
    gemini_client = None
    print("GEMINI_API_KEY was not found.")

Gemini API client is ready for gemma-4-31b-it


## Optional Extension A: Thinking Mode

Thinking Mode asks the hosted Gemma 4 model to spend extra reasoning effort.

Hosted Gemini API: use `gemma-4-31b-it` with `thinking_config=types.ThinkingConfig(thinking_level="high")`.

Local GGUF challenge: use Gemma 4 E2B with llama.cpp and place `<|think|>` at the start of the `system` message.

The Gemini API calls the system message `system_instruction`. In the local llama.cpp calls, the same idea appears as the `system` role.

This section compares valid JSON, status, root cause, next check, tool request, and time. It does not display hidden reasoning.

In [35]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
thinking_rows = []

api_modes = ["hosted base", "hosted thinking"]

for api_mode in api_modes:
    if gemini_client is None:
        print("Gemini API call skipped because no API key was loaded.")
    else:
        start_time = time.time()

        api_config = types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_json_schema=api_json_schema,
            temperature=0.1,
        )

        if api_mode == "hosted thinking":
            api_config = types.GenerateContentConfig(
                system_instruction=instructions,
                thinking_config=types.ThinkingConfig(thinking_level="high"),
                response_mime_type="application/json",
                response_json_schema=api_json_schema,
                temperature=0.1,
            )

        api_response = gemini_client.models.generate_content(
            model=gemma_api_model,
            contents=prompt,
            config=api_config,
        )

        elapsed_time = time.time() - start_time
        api_result = json.loads(api_response.text)

        thinking_rows.append({
            "Mode": api_mode,
            "Valid JSON": True,
            "Status": api_result.get("status", ""),
            "Likely Cause": api_result.get("likely_cause", ""),
            "Next Check": api_result.get("next_check", ""),
            "Tool": api_result.get("tool_name", ""),
            "Seconds": round(elapsed_time, 2),
        })

local_modes = ["local base", "local thinking"]

for local_mode in local_modes:
    local_instructions = instructions

    if local_mode == "local thinking":
        local_instructions = "<|think|>\\n" + instructions

    start_time = time.time()

    local_response = model.create_chat_completion(
        messages=[
            {"role": "system", "content": local_instructions},
            {"role": "user", "content": prompt}
        ],
        max_tokens=600,
        temperature=0.1,
        response_format=json_mode,
    )

    elapsed_time = time.time() - start_time
    local_text = local_response["choices"][0]["message"]["content"].strip()
    valid_json = local_text.startswith("{") and local_text.endswith("}")

    if valid_json:
        local_result = json.loads(local_text)
        local_status = local_result.get("status", "")
        local_likely_cause = local_result.get("likely_cause", "")
        local_next_check = local_result.get("next_check", "")
        local_tool = local_result.get("tool_name", "")
    else:
        local_status = "not valid JSON"
        local_likely_cause = ""
        local_next_check = ""
        local_tool = ""

    thinking_rows.append({
        "Mode": local_mode,
        "Valid JSON": valid_json,
        "Status": local_status,
        "Likely Cause": local_likely_cause,
        "Next Check": local_next_check,
        "Tool": local_tool,
        "Seconds": round(elapsed_time, 2),
    })

thinking_table = pd.DataFrame(thinking_rows)
thinking_table

,Mode,Valid JSON,Status,Likely Cause,Next Check,Tool,Seconds
0,hosted base,True,ready,The np.arange function in Python/NumPy exclude...,Check the documentation for np.arange or test ...,run_check,6.59
1,hosted thinking,True,ready,"The np.arange(0, n, 1) function generates a se...",Check the documentation for np.arange or print...,run_check,6.41
2,local base,True,ready,"The `np.arange(0, n, 1)` function generates nu...",Check the arguments passed to `np.arange` to s...,run_check,15.33
3,local thinking,True,ready,The loop condition in `np.arange` might be exc...,"Inspect the arguments passed to `np.arange(0, ...",run_check,13.56


### Extension Checkpoint A

Did Thinking Mode change the status, root cause, next check, tool request, or time?

*Double click to type your answer here*

## Optional Extension B: API Tool Calling

This extension shows hosted Gemma 4 function calling through the Gemini API.

The same rule still applies: the model does not run the tool by itself. The model only requests a function call. The notebook checks the request and runs the prepared Python check.

| Step | What happens |
|---|---|
| 1 | Send a function declaration to the model |
| 2 | Check whether the model returned a function call |
| 3 | Validate the function name and `case_name` |
| 4 | Run the allowed prepared check in Python |

Local E2B challenge: llama.cpp can be prompted with the same idea, but local tool calling may return text or a tool-like object depending on the runtime.

In [36]:
# RUN THIS CELL
run_check_declaration = {
    "name": "run_check",
    "description": "Run one prepared classroom debugging check.",
    "parameters": {
        "type": "object",
        "properties": {
            "case_name": {"type": "string"}
        },
        "required": ["case_name"]
    }
}

tool_prompt = f"""
Use the run_check tool before giving guidance.
Selected case: {selected_case["name"]}

Problem:
{selected_case["problem"]}

Expected:
{selected_case["expected"]}

Actual:
{selected_case["actual"]}

Code:
```python
{selected_case["code"]}
```

Tried:
{selected_case["tried"]}
""".strip()

if gemini_client is None:
    print("Gemini API call skipped because no API key was loaded.")
else:
    tools = types.Tool(function_declarations=[run_check_declaration])
    tool_config = types.GenerateContentConfig(
        system_instruction="You are a Gemma 4 debugging guide. Use tools only for evidence checks.",
        tools=[tools],
        temperature=0.1,
    )

    api_tool_response = gemini_client.models.generate_content(
        model=gemma_api_model,
        contents=tool_prompt,
        config=tool_config,
    )

    api_tool_call = None
    for part in api_tool_response.candidates[0].content.parts:
        if part.function_call:
            api_tool_call = part.function_call

    if api_tool_call is None:
        print("No function call returned.")
        print(api_tool_response.text)
    else:
        requested_case_name = api_tool_call.args.get("case_name", "")

        print("Function to call:", api_tool_call.name)
        print("Arguments:", api_tool_call.args)

        if api_tool_call.name == "run_check" and requested_case_name == selected_case["name"]:
            print("The app runs the prepared check:")
            print(selected_case["check_code"])

            check_program = """
import numpy as np
""" + "\n" + selected_case["check_code"]

            completed = subprocess.run(
                [sys.executable, "-I", "-c", check_program],
                capture_output=True,
                text=True,
                timeout=3,
            )

            api_check_result = {
                "ok": completed.returncode == 0,
                "stdout": completed.stdout,
                "stderr": completed.stderr
            }
            print(json.dumps(api_check_result, indent=2))
        else:
            print("The tool call did not match the allowed classroom check.")

local_tool_response = model.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are a Gemma 4 debugging guide. Request run_check when evidence would help."},
        {"role": "user", "content": tool_prompt}
    ],
    tools=[{
        "type": "function",
        "function": run_check_declaration
    }],
    tool_choice="auto",
    max_tokens=600,
    temperature=0.1,
)

print("Local E2B tool-calling challenge output:")
print(json.dumps(local_tool_response["choices"][0]["message"], indent=2))


Function to call: run_check
Arguments: {'case_name': 'Loop boundary'}
The app runs the prepared check:
print("Values generated by np.arange(0, 3, 1):")
print(np.arange(0, 3, 1))
{
  "ok": true,
  "stdout": "Values generated by np.arange(0, 3, 1):\n[0 1 2]\n",
  "stderr": ""
}
Local E2B tool-calling challenge output:
{
  "role": "assistant",
  "content": "<|tool_call>call:run_check{case_name:<|\"|>Loop boundary<|\"|>}<tool_call|>"
}


### Extension Checkpoint B

In API tool calling, who controls each part?

1. Who may request the tool call?
2. Who checks the tool name and arguments?
3. Who runs the Python code?

*Double click to type your answer here*


## Optional Extension C: Multimodal Input

A multimodal model can read more than one kind of input.

This extension sends `debugging_screenshot.png` to hosted Gemma 4 31B and asks the model to extract the bug report fields.

The flow is:

1. Screenshot to structured fields.
2. Structured fields to the same core workflow.

The image model should extract the bug report. It should not solve the bug yet.

Local E2B challenge: Gemma 4 E2B can be multimodal, but this text-only GGUF setup does not load an image runtime.


In [37]:
# RUN THIS CELL
screenshot_path = "debugging_screenshot.png"

screenshot_extraction_schema = {
    "type": "object",
    "properties": {
        "problem": {"type": "string"},
        "expected": {"type": "string"},
        "actual": {"type": "string"},
        "code": {"type": "string"},
        "tried": {"type": "string"}
    },
    "required": ["problem", "expected", "actual", "code", "tried"]
}

extraction_prompt = """
Read this debugging report image.
Extract exactly these lowercase keys: problem, expected, actual, code, tried.
Return valid JSON only.
Do not solve the bug.
""".strip()

if gemini_client is None or not os.path.exists(screenshot_path):
    screenshot_fields = {
        "problem": selected_case["problem"],
        "expected": selected_case["expected"],
        "actual": selected_case["actual"],
        "code": selected_case["code"],
        "tried": selected_case["tried"]
    }
    print("Using sample extracted fields from the selected case.")
else:
    uploaded_image = gemini_client.files.upload(file=screenshot_path)

    extraction_response = gemini_client.models.generate_content(
        model=gemma_api_model,
        contents=[uploaded_image, extraction_prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_json_schema=screenshot_extraction_schema,
            temperature=0.1,
        ),
    )

    screenshot_fields = json.loads(extraction_response.text)

print(json.dumps(screenshot_fields, indent=2))


{
  "problem": "I want to write a function that sums the squares from 0 through n.",
  "expected": "The sum should include n in the squared terms. For n = 3, the answer should include 3 squared.",
  "actual": "The current result does not include n. It stops one number too early.",
  "code": "def sum_squares(n):\n    total = 0\n    for i in np.arange(0, n, 1):\n        total = total + i ** 2\n    return total",
  "tried": "I checked the formula, but I did not check the exact values generated by the loop."
}


Send the extracted fields through the debugging workflow.

The image has become structured text, so the same workflow can use it.


In [38]:
# RUN THIS CELL: IT MAY TAKE A COUPLE OF MINUTES
screenshot_prompt = f"""
Please help me debug my code.
Do not give me the final corrected code.
All five report fields are present, so use status = "ready".

Problem:
{screenshot_fields["problem"]}

Expected:
{screenshot_fields["expected"]}

Actual:
{screenshot_fields["actual"]}

Code:
```python
{screenshot_fields["code"]}
```

Tried:
{screenshot_fields["tried"]}
""".strip()

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": screenshot_prompt}
]

response = model.create_chat_completion(
    messages=messages,
    max_tokens=600,
    temperature=0.1,
    response_format=json_mode,
)

screenshot_raw = response["choices"][0]["message"]["content"]
screenshot_result = json.loads(screenshot_raw)

screenshot_result["status"] = "ready"
screenshot_result["need"] = "none"

if screenshot_result.get("tool_name", "") == "":
    screenshot_result["tool_name"] = "run_check"

print(json.dumps(screenshot_result, indent=2))

{
  "status": "ready",
  "known": "The loop in your function stops before including the value 'n' in the calculation.",
  "need": "none",
  "likely_cause": "The range used in the `for` loop is exclusive of the upper bound.",
  "next_check": "Inspect the arguments passed to `np.arange(0, n, 1)` to see how the loop iterates.",
  "tool_name": "run_check",
  "hint": "Consider how the stop value in a range function affects the last number included.",
  "verify": "Test the function with a small value like n=3 and manually trace what values 'i' takes inside the loop.",
  "reflect": "The issue lies in the range definition of the loop, specifically how it handles the upper limit 'n'."
}


### Extension Checkpoint C

Why should the image model extract fields before the debugging assistant gives advice?

*Double click to type your answer here*


## Summary

You built a local debugging assistant with Gemma 4 E2B.

The assistant supports this habit:

1. Observe the problem.
2. Name a likely root cause.
3. Run a diagnostic check.
4. Verify the fix.
5. Reflect on the habit.

You also practiced two levels of tool use:

1. A beginner tool loop in the local notebook.
2. Hosted Gemma 4 API extensions for Thinking Mode, Multimodal Input, and Tool Calling.

The main lesson is that the model should support structured debugging practice. It should not replace student verification.
